In [27]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

plt.rcParams["figure.figsize"] = (14, 5)

In [2]:
# ============================================
# LOAD XEMA SILVER DATASET
# ============================================

PATH = "/Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/03_Especialitzacio/Project/thermal-wind-prediction/data/processed/xema_silver.parquet"

df = pd.read_parquet(PATH)

print("Shape:", df.shape)

df.head()

Shape: (4791830, 12)


,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,CODI_ESTAT,CODI_BASE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS,VALOR_CLEAN
0,WU320101090000,WU,32,2009-01-01,V,SH,Temperatura,°C,T,DAT,1,10.2
1,WU330101090000,WU,33,2009-01-01,V,SH,Humitat relativa,%,HR,DAT,0,89.0
2,WU340101090000,WU,34,2009-01-01,V,SH,Pressió atmosfèrica,hPa,P,DAT,1,1016.0
3,WU350101090000,WU,35,2009-01-01,V,SH,Precipitació,mm,PPT,DAT,1,0.0
4,WU010101090000,WU,1,2009-01-01,V,HO,Pressió atmosfèrica (legacy),hPa,P_LEGACY,DAT,0,1017.0


In [3]:
# ============================================
# BASIC INFO
# ============================================

df.info()

print("\nMemory usage (MB):")
print(df.memory_usage(deep=True).sum() / 1024**2)

print("\nColumns:")
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 4791830 entries, 0 to 4791829
Data columns (total 12 columns):
 #   Column          Dtype         
---  ------          -----         
 0   ID              str           
 1   CODI_ESTACIO    category      
 2   CODI_VARIABLE   int8          
 3   DATA_LECTURA    datetime64[us]
 4   CODI_ESTAT      category      
 5   CODI_BASE       category      
 6   NOM_VARIABLE    category      
 7   UNITAT          category      
 8   ACRONIM         category      
 9   CODI_TIPUS_VAR  category      
 10  DECIMALS        int8          
 11  VALOR_CLEAN     float32       
dtypes: category(7), datetime64[us](1), float32(1), int8(2), str(1)
memory usage: 196.5 MB

Memory usage (MB):
196.50426959991455

Columns:
['ID', 'CODI_ESTACIO', 'CODI_VARIABLE', 'DATA_LECTURA', 'CODI_ESTAT', 'CODI_BASE', 'NOM_VARIABLE', 'UNITAT', 'ACRONIM', 'CODI_TIPUS_VAR', 'DECIMALS', 'VALOR_CLEAN']


In [4]:
# ============================================
# DATETIME CHECK
# ============================================

print(df.index)

print(type(df.index))

print(df.index.min())
print(df.index.max())

RangeIndex(start=0, stop=4791830, step=1)
<class 'pandas.RangeIndex'>
0
4791829


In [5]:
# ============================================
# SET DATETIME INDEX
# ============================================

df["DATA_LECTURA"] = pd.to_datetime(df["DATA_LECTURA"])

df = df.sort_values("DATA_LECTURA")

df = df.set_index("DATA_LECTURA")

print(df.index)

df.head()

DatetimeIndex(['2009-01-01 00:00:00', '2009-01-01 00:00:00', '2009-01-01 00:00:00', '2009-01-01 00:00:00', '2009-01-01 00:00:00',
               '2009-01-01 00:00:00', '2009-01-01 00:00:00', '2009-01-01 00:00:00', '2009-01-01 00:00:00', '2009-01-01 00:00:00',
               ...
               '2026-06-14 08:00:00', '2026-06-14 08:00:00', '2026-06-14 08:00:00', '2026-06-14 08:00:00', '2026-06-14 08:00:00',
               '2026-06-14 08:00:00', '2026-06-14 08:00:00', '2026-06-14 08:00:00', '2026-06-14 08:00:00', '2026-06-14 08:00:00'],
              dtype='datetime64[us]', name='DATA_LECTURA', length=4791830, freq=None)


,ID,CODI_ESTACIO,CODI_VARIABLE,CODI_ESTAT,CODI_BASE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS,VALOR_CLEAN
DATA_LECTURA,,,,,,,,,,,
2009-01-01,WU320101090000,WU,32,V,SH,Temperatura,°C,T,DAT,1,10.2
2009-01-01,WU330101090000,WU,33,V,SH,Humitat relativa,%,HR,DAT,0,89.0
2009-01-01,WU340101090000,WU,34,V,SH,Pressió atmosfèrica,hPa,P,DAT,1,1016.0
2009-01-01,WU350101090000,WU,35,V,SH,Precipitació,mm,PPT,DAT,1,0.0
2009-01-01,WU010101090000,WU,1,V,HO,Pressió atmosfèrica (legacy),hPa,P_LEGACY,DAT,0,1017.0


In [6]:
# ============================================
# DUPLICATED TIMESTAMPS
# ============================================

duplicated = df.index.duplicated().sum()

print("Duplicated timestamps:", duplicated)

Duplicated timestamps: 4486216


In [7]:
# ============================================
# PIVOT TO WIDE FORMAT
# ============================================

pivot_df = df.pivot_table(
    index="DATA_LECTURA",
    columns="NOM_VARIABLE",
    values="VALOR_CLEAN",
    aggfunc="mean"
)

pivot_df = pivot_df.sort_index()

pivot_df.head()

NOM_VARIABLE,Direcció de la ratxa màxima del vent a 6 m,Direcció del vent a 6 m (m. 1),Humitat relativa,Humitat relativa màxima,Humitat relativa mínima,Irradiància solar global,Precipitació,Precipitació màxima en 1 minut,Pressió atmosfèrica,Pressió atmosfèrica (legacy),Pressió atmosfèrica mínima,Ratxa màxima del vent a 6 m,Temperatura,Temperatura màxima,Temperatura mínima,Velocitat del vent a 6 m (esc.)
DATA_LECTURA,,,,,,,,,,,,,,,,
2009-01-01 00:00:00,NaN,NaN,89.0,NaN,88.0,0.0,0.0,NaN,1016.0,1017.0,NaN,4.2,10.2,10.4,9.9,NaN
2009-01-01 00:30:00,NaN,NaN,89.0,NaN,87.0,0.0,0.0,NaN,1016.0,NaN,NaN,4.6,10.1,10.4,9.9,NaN
2009-01-01 01:00:00,NaN,NaN,84.0,NaN,83.0,0.0,0.0,NaN,1016.0,NaN,NaN,4.7,10.5,10.6,10.4,NaN
2009-01-01 01:30:00,NaN,NaN,85.0,NaN,84.0,0.0,0.0,NaN,1015.0,NaN,NaN,4.7,10.4,10.4,10.3,NaN
2009-01-01 02:00:00,NaN,NaN,82.0,NaN,80.0,0.0,0.0,NaN,1015.0,NaN,NaN,4.0,10.5,10.5,10.4,NaN


In [8]:
# ============================================
# PIVOT INFO
# ============================================

print(pivot_df.shape)

display(pivot_df.head())

print("\nColumns:")
print(pivot_df.columns.tolist())

(305614, 16)


NOM_VARIABLE,Direcció de la ratxa màxima del vent a 6 m,Direcció del vent a 6 m (m. 1),Humitat relativa,Humitat relativa màxima,Humitat relativa mínima,Irradiància solar global,Precipitació,Precipitació màxima en 1 minut,Pressió atmosfèrica,Pressió atmosfèrica (legacy),Pressió atmosfèrica mínima,Ratxa màxima del vent a 6 m,Temperatura,Temperatura màxima,Temperatura mínima,Velocitat del vent a 6 m (esc.)
DATA_LECTURA,,,,,,,,,,,,,,,,
2009-01-01 00:00:00,NaN,NaN,89.0,NaN,88.0,0.0,0.0,NaN,1016.0,1017.0,NaN,4.2,10.2,10.4,9.9,NaN
2009-01-01 00:30:00,NaN,NaN,89.0,NaN,87.0,0.0,0.0,NaN,1016.0,NaN,NaN,4.6,10.1,10.4,9.9,NaN
2009-01-01 01:00:00,NaN,NaN,84.0,NaN,83.0,0.0,0.0,NaN,1016.0,NaN,NaN,4.7,10.5,10.6,10.4,NaN
2009-01-01 01:30:00,NaN,NaN,85.0,NaN,84.0,0.0,0.0,NaN,1015.0,NaN,NaN,4.7,10.4,10.4,10.3,NaN
2009-01-01 02:00:00,NaN,NaN,82.0,NaN,80.0,0.0,0.0,NaN,1015.0,NaN,NaN,4.0,10.5,10.5,10.4,NaN



Columns:
['Direcció de la ratxa màxima del vent a 6 m', 'Direcció del vent a 6 m (m. 1)', 'Humitat relativa', 'Humitat relativa màxima', 'Humitat relativa mínima', 'Irradiància solar global', 'Precipitació', 'Precipitació màxima en 1 minut', 'Pressió atmosfèrica', 'Pressió atmosfèrica (legacy)', 'Pressió atmosfèrica mínima', 'Ratxa màxima del vent a 6 m', 'Temperatura', 'Temperatura màxima', 'Temperatura mínima', 'Velocitat del vent a 6 m (esc.)']


In [9]:
# ============================================
# DUPLICATED TIMESTAMPS
# ============================================

print(
    pivot_df.index.duplicated().sum()
)

0


In [10]:
print("Start:", pivot_df.index.min())
print("End:", pivot_df.index.max())

Start: 2009-01-01 00:00:00
End: 2026-06-14 08:00:00


In [11]:
time_diff = pivot_df.index.to_series().diff()

display(
    time_diff.value_counts().head(20)
)

DATA_LECTURA
0 days 00:30:00    305591
0 days 01:30:00         6
0 days 01:00:00         5
0 days 02:30:00         3
2 days 16:00:00         1
0 days 08:00:00         1
1 days 11:00:00         1
0 days 05:00:00         1
0 days 20:30:00         1
0 days 02:00:00         1
0 days 04:00:00         1
0 days 04:30:00         1
Name: count, dtype: int64

In [12]:
# ============================================
# FILTER DATE RANGE
# ============================================

pivot_df = pivot_df.loc[
    "2010-01-01":"2025-12-31"
].copy()

print("Shape:", pivot_df.shape)

print("\nStart:")
print(pivot_df.index.min())

print("\nEnd:")
print(pivot_df.index.max())

Shape: (280205, 16)

Start:
2010-01-01 00:00:00

End:
2025-12-31 23:30:00


In [13]:
# ============================================
# MISSING VALUES %
# ============================================

missing_pct = (
    pivot_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

display(
    missing_pct.round(2)
)

NOM_VARIABLE
Direcció del vent a 6 m (m. 1)                0.18
Direcció de la ratxa màxima del vent a 6 m    0.18
Temperatura                                   0.07
Temperatura màxima                            0.07
Temperatura mínima                            0.07
Humitat relativa                              0.05
Humitat relativa mínima                       0.05
Humitat relativa màxima                       0.05
Pressió atmosfèrica mínima                    0.03
Pressió atmosfèrica (legacy)                  0.03
Pressió atmosfèrica                           0.03
Precipitació màxima en 1 minut                0.02
Irradiància solar global                      0.02
Ratxa màxima del vent a 6 m                   0.02
Velocitat del vent a 6 m (esc.)               0.02
Precipitació                                  0.00
dtype: float64

In [14]:
# ============================================
# CREATE FULL 30MIN GRID
# ============================================

full_index = pd.date_range(
    start=pivot_df.index.min(),
    end=pivot_df.index.max(),
    freq="30min"
)

pivot_df = pivot_df.reindex(full_index)

pivot_df.index.name = "DATA_LECTURA"

In [15]:
missing_pct = (
    pivot_df
    .isna()
    .mean()
    * 100
)

display(
    missing_pct.round(2)
)

NOM_VARIABLE
Direcció de la ratxa màxima del vent a 6 m    0.29
Direcció del vent a 6 m (m. 1)                0.29
Humitat relativa                              0.16
Humitat relativa màxima                       0.16
Humitat relativa mínima                       0.16
Irradiància solar global                      0.13
Precipitació                                  0.11
Precipitació màxima en 1 minut                0.13
Pressió atmosfèrica                           0.14
Pressió atmosfèrica (legacy)                  0.14
Pressió atmosfèrica mínima                    0.14
Ratxa màxima del vent a 6 m                   0.13
Temperatura                                   0.18
Temperatura màxima                            0.18
Temperatura mínima                            0.18
Velocitat del vent a 6 m (esc.)               0.13
dtype: float64

In [16]:
# ============================================
# GAP SIZE ANALYSIS
# ============================================

missing_mask = pivot_df["Temperatura"].isna()

groups = (
    missing_mask != missing_mask.shift()
).cumsum()

gap_sizes = (
    missing_mask.groupby(groups)
    .sum()
)

gap_sizes = gap_sizes[gap_sizes > 0]

display(
    gap_sizes.describe()
)

print("\nLargest gaps:")
display(
    gap_sizes.sort_values(ascending=False).head(20)
)

count     25.000000
mean      20.280000
std       36.035307
min        1.000000
25%        2.000000
50%        4.000000
75%        9.000000
max      127.000000
Name: Temperatura, dtype: float64


Largest gaps:


Temperatura
8     127
6     108
4      82
12     69
26     40
10     15
14      9
50      8
28      7
48      7
18      4
20      4
2       4
16      4
32      3
36      2
38      2
40      2
44      2
46      2
Name: Temperatura, dtype: int64

In [17]:
# ============================================
# CONTROLLED INTERPOLATION
# ============================================

clean_df = pivot_df.copy()

clean_df = clean_df.interpolate(
    method="time",
    limit=4,          # máximo 4 timestamps
    limit_direction="both"
)

In [18]:
missing_after_interp = (
    clean_df
    .isna()
    .mean()
    * 100
)

display(
    missing_after_interp.round(2)
)

NOM_VARIABLE
Direcció de la ratxa màxima del vent a 6 m    0.23
Direcció del vent a 6 m (m. 1)                0.23
Humitat relativa                              0.11
Humitat relativa màxima                       0.11
Humitat relativa mínima                       0.11
Irradiància solar global                      0.09
Precipitació                                  0.08
Precipitació màxima en 1 minut                0.08
Pressió atmosfèrica                           0.10
Pressió atmosfèrica (legacy)                  0.10
Pressió atmosfèrica mínima                    0.10
Ratxa màxima del vent a 6 m                   0.08
Temperatura                                   0.14
Temperatura màxima                            0.13
Temperatura mínima                            0.13
Velocitat del vent a 6 m (esc.)               0.08
dtype: float64

In [19]:
# ============================================
# RESAMPLING CONFIG
# ============================================

agg_dict = {

    # TEMPERATURE
    "Temperatura": "mean",
    "Temperatura màxima": "max",
    "Temperatura mínima": "min",

    # HUMIDITY
    "Humitat relativa": "mean",
    "Humitat relativa màxima": "max",
    "Humitat relativa mínima": "min",

    # PRESSURE
    "Pressió atmosfèrica": "mean",
    "Pressió atmosfèrica (legacy)": "mean",
    "Pressió atmosfèrica mínima": "min",

    # WIND
    "Velocitat del vent a 6 m (esc.)": "mean",
    "Ratxa màxima del vent a 6 m": "max",

    # WIND DIRECTION
    "Direcció del vent a 6 m (m. 1)": "mean",
    "Direcció de la ratxa màxima del vent a 6 m": "mean",

    # RAIN
    "Precipitació": "sum",
    "Precipitació màxima en 1 minut": "max",

    # SOLAR
    "Irradiància solar global": "sum"
}

In [20]:
# ============================================
# RESAMPLE TO 1H
# ============================================

hourly_df = (
    clean_df
    .resample("1h")
    .agg(agg_dict)
)

print(hourly_df.shape)

hourly_df.head()

(140256, 16)


NOM_VARIABLE,Temperatura,Temperatura màxima,Temperatura mínima,Humitat relativa,Humitat relativa màxima,Humitat relativa mínima,Pressió atmosfèrica,Pressió atmosfèrica (legacy),Pressió atmosfèrica mínima,Velocitat del vent a 6 m (esc.),Ratxa màxima del vent a 6 m,Direcció del vent a 6 m (m. 1),Direcció de la ratxa màxima del vent a 6 m,Precipitació,Precipitació màxima en 1 minut,Irradiància solar global
DATA_LECTURA,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,11.80,12.1,11.3,48.0,49.0,46.0,989.5,990.5,989.0,7.40,16.900000,255.5,246.0,0.2,0.2,0.0
2010-01-01 01:00:00,10.90,11.3,10.3,49.0,51.0,47.0,990.0,991.5,989.0,7.80,17.200001,265.5,266.0,0.0,0.0,0.0
2010-01-01 02:00:00,10.30,10.3,10.2,49.5,51.0,48.0,990.0,991.5,989.0,7.95,15.900000,268.5,262.5,0.0,0.0,0.0
2010-01-01 03:00:00,9.85,10.2,9.6,47.0,48.0,47.0,990.5,992.0,990.0,8.00,16.299999,265.5,261.0,0.0,0.0,0.0
2010-01-01 04:00:00,9.50,9.6,9.4,46.0,47.0,45.0,991.0,992.0,990.0,7.15,15.900000,260.0,255.0,0.0,0.0,0.0


In [21]:
# ============================================
# FINAL MISSING %
# ============================================

final_missing = (
    hourly_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

display(
    final_missing.round(2)
)

NOM_VARIABLE
Direcció del vent a 6 m (m. 1)                0.23
Direcció de la ratxa màxima del vent a 6 m    0.23
Temperatura                                   0.14
Temperatura màxima                            0.13
Temperatura mínima                            0.13
Humitat relativa                              0.11
Humitat relativa mínima                       0.11
Humitat relativa màxima                       0.11
Pressió atmosfèrica                           0.10
Pressió atmosfèrica mínima                    0.10
Pressió atmosfèrica (legacy)                  0.10
Precipitació màxima en 1 minut                0.08
Velocitat del vent a 6 m (esc.)               0.08
Ratxa màxima del vent a 6 m                   0.08
Precipitació                                  0.00
Irradiància solar global                      0.00
dtype: float64

In [22]:
# ============================================
# MISSING BY YEAR
# ============================================

yearly_missing = (
    hourly_df["Temperatura"]
    .isna()
    .groupby(hourly_df.index.year)
    .mean()
    * 100
)

display(
    yearly_missing.round(2)
)

DATA_LECTURA
2010    0.00
2011    0.00
2012    0.00
2013    0.00
2014    0.00
2015    0.41
2016    0.00
2017    0.56
2018    0.00
2019    0.00
2020    1.05
2021    0.00
2022    0.00
2023    0.00
2024    0.18
2025    0.00
Name: Temperatura, dtype: float64

In [31]:
# ============================================
# VALIDATION FLAG
# ============================================

df["IS_VALIDATED"] = (
    df["CODI_ESTAT"] == "V"
)

# ============================================
# SAVE 30MIN MASTER DATASET
# ============================================

pivot_df.to_parquet(
    "/Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/03_Especialitzacio/Project/thermal-wind-prediction/data/processed/gold/xema_30min_master.parquet"
)

print("30min master dataset exported")

# ============================================
# RESAMPLING CONFIG
# ============================================

agg_dict = {

    # TEMPERATURE
    "Temperatura": "mean",
    "Temperatura màxima": "max",
    "Temperatura mínima": "min",

    # HUMIDITY
    "Humitat relativa": "mean",
    "Humitat relativa màxima": "max",
    "Humitat relativa mínima": "min",

    # PRESSURE
    "Pressió atmosfèrica": "mean",
    "Pressió atmosfèrica (legacy)": "mean",
    "Pressió atmosfèrica mínima": "min",

    # WIND
    "Velocitat del vent a 6 m (esc.)": "mean",
    "Ratxa màxima del vent a 6 m": "max",

    # WIND DIRECTION
    "Direcció del vent a 6 m (m. 1)": "mean",
    "Direcció de la ratxa màxima del vent a 6 m": "mean",

    # RAIN
    "Precipitació": "sum",
    "Precipitació màxima en 1 minut": "max",

    # SOLAR
    "Irradiància solar global": "sum"
}

# ============================================
# CONTROLLED INTERPOLATION
# ============================================

clean_df = pivot_df.copy()

clean_df = clean_df.interpolate(
    method="time",
    limit=4,
    limit_direction="both"
)

# ============================================
# RESAMPLE TO 1H
# ============================================

hourly_df = (
    clean_df
    .resample("1h")
    .agg(agg_dict)
)

print("\nHourly dataset shape:")
print(hourly_df.shape)

# ============================================
# WIND VECTOR COMPONENTS
# ============================================

direction_rad = np.deg2rad(
    hourly_df["Direcció del vent a 6 m (m. 1)"]
)

speed = hourly_df[
    "Velocitat del vent a 6 m (esc.)"
]

hourly_df["wind_u"] = (
    speed * np.cos(direction_rad)
)

hourly_df["wind_v"] = (
    speed * np.sin(direction_rad)
)

# ============================================
# DROP REDUNDANT COLUMNS
# ============================================

hourly_df = hourly_df.drop(columns=[
    "Direcció del vent a 6 m (m. 1)",
    "Direcció de la ratxa màxima del vent a 6 m",
    "Pressió atmosfèrica (legacy)"
])

# ============================================
# FINAL MISSING %
# ============================================

final_missing = (
    hourly_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

print("\nFinal Missing %:")
display(
    final_missing.round(3)
)

# ============================================
# FINAL EXPORT
# ============================================

from pathlib import Path

# --------------------------------------------
# CREATE OUTPUT DIRECTORY
# --------------------------------------------

output_dir = Path(
    "/Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/03_Especialitzacio/Project/thermal-wind-prediction/data/gold"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------
# OUTPUT FILES
# --------------------------------------------

OUTPUT_PARQUET = output_dir / "xema_hourly_model_ready.parquet"

OUTPUT_CSV = output_dir / "xema_hourly_model_ready.csv"

# --------------------------------------------
# EXPORT FILES
# --------------------------------------------

hourly_df.to_parquet(OUTPUT_PARQUET)

hourly_df.to_csv(OUTPUT_CSV)

print("\nExports completed successfully")

print(f"\nParquet:\n{OUTPUT_PARQUET}")

print(f"\nCSV:\n{OUTPUT_CSV}")

# ============================================
# FINAL INFO
# ============================================

print("\nFinal dataset shape:")
print(hourly_df.shape)

print("\nDate range:")
print(hourly_df.index.min())
print(hourly_df.index.max())

display(hourly_df.head())

30min master dataset exported

Hourly dataset shape:
(140256, 16)

Final Missing %:


NOM_VARIABLE
wind_u                             0.227
wind_v                             0.227
Temperatura                        0.138
Temperatura màxima                 0.131
Temperatura mínima                 0.130
Humitat relativa                   0.112
Humitat relativa mínima            0.112
Humitat relativa màxima            0.109
Pressió atmosfèrica                0.102
Pressió atmosfèrica mínima         0.102
Precipitació màxima en 1 minut     0.079
Velocitat del vent a 6 m (esc.)    0.077
Ratxa màxima del vent a 6 m        0.077
Precipitació                       0.000
Irradiància solar global           0.000
dtype: float64


Exports completed successfully

Parquet:
/Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/03_Especialitzacio/Project/thermal-wind-prediction/data/gold/xema_hourly_model_ready.parquet

CSV:
/Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/03_Especialitzacio/Project/thermal-wind-prediction/data/gold/xema_hourly_model_ready.csv

Final dataset shape:
(140256, 15)

Date range:
2010-01-01 00:00:00
2025-12-31 23:00:00


NOM_VARIABLE,Temperatura,Temperatura màxima,Temperatura mínima,Humitat relativa,Humitat relativa màxima,Humitat relativa mínima,Pressió atmosfèrica,Pressió atmosfèrica mínima,Velocitat del vent a 6 m (esc.),Ratxa màxima del vent a 6 m,Precipitació,Precipitació màxima en 1 minut,Irradiància solar global,wind_u,wind_v
DATA_LECTURA,,,,,,,,,,,,,,,
2010-01-01 00:00:00,11.80,12.1,11.3,48.0,49.0,46.0,989.5,989.0,7.40,16.900000,0.2,0.2,0.0,-1.852812,-7.164292
2010-01-01 01:00:00,10.90,11.3,10.3,49.0,51.0,47.0,990.0,989.0,7.80,17.200001,0.0,0.0,0.0,-0.611981,-7.775955
2010-01-01 02:00:00,10.30,10.3,10.2,49.5,51.0,48.0,990.0,989.0,7.95,15.900000,0.0,0.0,0.0,-0.208105,-7.947276
2010-01-01 03:00:00,9.85,10.2,9.6,47.0,48.0,47.0,990.5,990.0,8.00,16.299999,0.0,0.0,0.0,-0.627673,-7.975338
2010-01-01 04:00:00,9.50,9.6,9.4,46.0,47.0,45.0,991.0,990.0,7.15,15.900000,0.0,0.0,0.0,-1.241584,-7.041375


In [32]:
# ============================================
# EXPORT SAMPLE
# ============================================

sample_df = hourly_df.head(10)

sample_df.to_csv(
    "xema_sample_10rows.csv"
)

print("Sample exported")

Sample exported


In [33]:
pd.Series(df.dtypes).to_csv(
    "xema_dtypes.csv"
)